# Why does motifbreakR predict MPRA effects so poorly? Malinois and Kircher side by side

motifbreakR (our re-implementation, as in every earlier notebook: strong match > 0.85 for one allele, |alleleDiff| ≥
0.4, a FIMO hit of the same TF containing the variant, the 251 TFs with a function CAV) predicts the Malinois allelic
effects with Pearson 0.07 (0.14 on active elements) and AUROC 0.51–0.55 for clear effects; on the Kircher regions
the mean per-region Pearson is 0.12 (0.30 in the 4 HepG2/K562 regions). Candidate reasons, each checked here:

| | reason | check |
|---|---|---|
| 1 | little of Δ is predictable at all | share of the variance of Δ that is not measurement noise; agreement of Δ between cells |
| 2 | the real effects are not where motifbreakR looks | clear-effect rate: no motif hit / motif hit without a call / call; share of all clear effects with a call |
| 3 | inactive elements have nothing to lose | motifbreakR by element activity (Malinois) |
| 4 | a bigger motif change does not mean a bigger effect | \|Δ\| and direction agreement by \|alleleDiff\| fifths |
| 5 | not every TF matters (or some repress) | per-TF correlation of alleleDiff with Δ; TFs whose ChIP comes from this cell vs the other |
| 6 | the pick among several motifs | largest \|alleleDiff\| vs alleleDiff summed over calls vs variants with only one call |

**Clear effect.** Malinois: |Δ| ≥ 2 SE. Kircher: the strongest 15% of SNVs by |log2FC| within the region (the
definition of the earlier Kircher notebooks) and, if the MPRA file has a p-value column, also p < 1e-5.
CPU only, no fitting; reads `malinois/` and `fcav_pool/`. A few minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
# Section 1 — Malinois

## 1.1 Load

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, re
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{WORK_DIR}/mbr_diagnosis"
os.makedirs(OUT, exist_ok=True)
CELLS3 = ["HepG2", "K562", "SKNSH"]
ACTIVE_MIN, Z_CLEAR, MIN_TF = 1.0, 2.0, 100
pd.set_option("display.width", 250)
fmt = lambda x: f"{x:+.3f}"


def pear(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))


def auroc(lab, score):
    lab = np.asarray(lab, bool)
    return roc_auc_score(lab, score) if 0 < lab.sum() < len(lab) else np.nan


V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
H = pd.read_csv(f"{MAL}/hits.csv", usecols=["vi", "motif"])
CALLS = pd.read_csv(f"{MAL}/calls.csv")
PICK = CALLS.loc[CALLS["alleleDiff"].abs().sort_values(ascending=False, kind="stable").index].drop_duplicates("vi").set_index("vi")
V["has_hit"] = V.index.isin(H["vi"])
V["has_call"] = V.index.isin(CALLS["vi"])
V["ad"] = PICK["alleleDiff"].reindex(V.index)
V["tf"] = PICK["motif"].reindex(V.index)
V["n_calls"] = CALLS.groupby("vi").size().reindex(V.index).fillna(0).astype(int)
V["sum_ad"] = CALLS.groupby("vi")["alleleDiff"].sum().reindex(V.index)
for c in CELLS3:
    V[f"ok_{c}"] = V[f"d_{c}"].notna() & V[f"se_{c}"].notna()
    V[f"clear_{c}"] = V[f"ok_{c}"] & (V[f"d_{c}"].abs() / V[f"se_{c}"] >= Z_CLEAR)


def sub(c, s):
    return V[f"ok_{c}"] & ((V[f"act_{c}"] >= ACTIVE_MIN) if s == "active" else True)


print(f"{len(V):,} variants; {int(V['has_hit'].sum()):,} inside a FIMO hit of the 251 TFs; "
      f"{int(V['has_call'].sum()):,} with a motifbreakR call ({len(CALLS):,} calls)")

## 1.2 Reason 1: how much of Δ is predictable at all?

Δ = true effect + measurement noise, and the noise variance of each variant is SE². So the share of the variance of
Δ that is real effect ≈ 1 − mean(SE²) / var(Δ), and no predictor can reach a Pearson above about √(that share).
Second table: Pearson of Δ between cells (all variants, and variants active in both cells).

In [ ]:
rows = []
for c in CELLS3:
    for s in ("all", "active"):
        m = sub(c, s)
        y, se = V.loc[m, f"d_{c}"], V.loc[m, f"se_{c}"]
        share = 1 - (se ** 2).mean() / y.var()
        rows.append({"cell": c, "variants": s, "n": int(m.sum()), "share with a clear effect": V.loc[m, f"clear_{c}"].mean(),
                     "var(Δ)": y.var(), "mean SE²": (se ** 2).mean(), "real-effect share of var(Δ)": share,
                     "best possible Pearson ≈": np.sqrt(max(share, 0))})
T1 = pd.DataFrame(rows).set_index(["cell", "variants"])
print(T1.to_string(float_format=lambda x: f"{x:.3f}"))
D = V[[f"d_{c}" for c in CELLS3]]
act_all = (V[[f"act_{c}" for c in CELLS3]] >= ACTIVE_MIN).all(1)
print("\nPearson of Δ between cells: all variants (upper) / active in all three (lower)\n")
print(pd.concat([D.corr(), D[act_all].corr()], keys=["all", "active in all three"]).to_string(float_format=lambda x: f"{x:.3f}"))
T1.to_csv(f"{OUT}/malinois_ceiling.csv")

## 1.3 Reason 2: are the clear effects where motifbreakR looks?

In [ ]:
rows = []
cat = np.where(V["has_call"], "3 motifbreakR call", np.where(V["has_hit"], "2 motif hit, no call", "1 no motif hit"))
for c in CELLS3:
    for s in ("all", "active"):
        m = sub(c, s).values
        cl = V[f"clear_{c}"].values
        for k in sorted(set(cat)):
            mk = m & (cat == k)
            rows.append({"cell": c, "variants": s, "where": k[2:], "n": int(mk.sum()), "share of variants": mk.sum() / m.sum(),
                         "clear-effect rate": cl[mk].mean(), "share of all clear effects": cl[mk].sum() / cl[m].sum(),
                         "median |Δ|": np.median(np.abs(V[f"d_{c}"].values[mk]))})
T2 = pd.DataFrame(rows).set_index(["cell", "variants", "where"])
print(T2.to_string(float_format=lambda x: f"{x:.3f}"))
T2.to_csv(f"{OUT}/malinois_coverage.csv")

## 1.4 Reason 3: element activity

Fifths of the element's activity in that cell (max over the two alleles of log2FC). Per fifth: clear-effect rate of
all variants, and motifbreakR on its called variants (Pearson with Δ, AUROC for clear effects).

In [ ]:
rows = []
for c in CELLS3:
    m = V[f"ok_{c}"]
    q = pd.qcut(V.loc[m, f"act_{c}"].rank(method="first"), 5, labels=False)
    for b in range(5):
        g = V.loc[m][q == b]
        gc_ = g[g["has_call"]]
        rows.append({"cell": c, "activity fifth": b + 1, "activity range": f"{g[f'act_{c}'].min():+.2f} .. {g[f'act_{c}'].max():+.2f}",
                     "clear-effect rate": g[f"clear_{c}"].mean(), "called": len(gc_),
                     "Pearson motifbreakR": pear(gc_["ad"], gc_[f"d_{c}"]),
                     "AUROC motifbreakR": auroc(gc_[f"clear_{c}"], gc_["ad"].abs())})
T3 = pd.DataFrame(rows).set_index(["cell", "activity fifth"])
print(T3.to_string(float_format=fmt))
T3.to_csv(f"{OUT}/malinois_by_activity.csv")

## 1.5 Reason 4: does a bigger motif change give a bigger effect?

Called variants in fifths of |alleleDiff|: median |Δ|, clear-effect rate, and among clear effects the share whose
sign motifbreakR gets right (0.5 = chance).

In [ ]:
rows = []
for c in CELLS3:
    for s in ("all", "active"):
        g = V[sub(c, s) & V["has_call"]]
        q = pd.qcut(g["ad"].abs().rank(method="first"), 5, labels=False)
        for b in range(5):
            x = g[q == b]
            cl = x[x[f"clear_{c}"]]
            rows.append({"cell": c, "variants": s, "|alleleDiff| fifth": b + 1, "mean |alleleDiff|": x["ad"].abs().mean(),
                         "median |Δ|": x[f"d_{c}"].abs().median(), "clear-effect rate": x[f"clear_{c}"].mean(),
                         "sign right among clear": (np.sign(cl["ad"]) == np.sign(cl[f"d_{c}"])).mean()})
T4 = pd.DataFrame(rows).set_index(["cell", "variants", "|alleleDiff| fifth"])
print(T4.to_string(float_format=lambda x: f"{x:.3f}"))
T4.to_csv(f"{OUT}/malinois_by_alleleDiff.csv")

## 1.6 Reason 5: does every TF matter?

For each TF picked for at least 100 variants: Pearson of its alleleDiff with Δ over those variants, and z (Fisher).
Negative = the measured effect goes against the motif change (a repressor, or noise). The TF's ChIP source is the
share of its bound sites from HepG2 (the rest from K562) in the CAV's site file.

In [ ]:
SH = {}
for m_ in V["tf"].dropna().unique():
    rec = json.load(open(f"{POOL}/sites/{m_}.json")).get("records", [])
    cells = [r.get("cell") for r in rec if r.get("cls") == 1 and r.get("cell")]
    SH[m_] = np.mean([x == "HepG2" for x in cells]) if cells else np.nan
print(f"ChIP source known for {np.isfinite(list(SH.values())).sum()} of {len(SH)} TFs")
rows, per_tf = [], []
for c in CELLS3:
    for s in ("all", "active"):
        g = V[sub(c, s) & V["has_call"]]
        for tf, x in g.groupby("tf"):
            if len(x) >= MIN_TF and x["ad"].std() > 0:
                r = pear(x["ad"], x[f"d_{c}"])
                per_tf.append({"cell": c, "variants": s, "TF": tf, "n": len(x), "r": r,
                               "z": np.arctanh(r) * np.sqrt(len(x) - 3), "ChIP share HepG2": SH.get(tf, np.nan)})
PT = pd.DataFrame(per_tf)
for (c, s), x in PT.groupby(["cell", "variants"], sort=False):
    row = {"cell": c, "variants": s, "TFs": len(x), "median r": x["r"].median(), "share r < 0": (x["r"] < 0).mean(),
           "share z >= 2": (x["z"] >= 2).mean(), "share z <= -2": (x["z"] <= -2).mean()}
    if c in ("HepG2", "K562"):
        own = x["ChIP share HepG2"] >= 0.5 if c == "HepG2" else x["ChIP share HepG2"] < 0.5
        known = x["ChIP share HepG2"].notna()
        row.update({"median r, ChIP mostly from this cell": x.loc[own & known, "r"].median(),
                    "median r, ChIP mostly from the other cell": x.loc[~own & known, "r"].median(),
                    "TFs (this / other)": f"{int((own & known).sum())} / {int((~own & known).sum())}"})
    rows.append(row)
T5 = pd.DataFrame(rows).set_index(["cell", "variants"])
print(T5.to_string(float_format=fmt))
for c in CELLS3:
    x = PT[(PT["cell"] == c) & (PT["variants"] == "all")].sort_values("z")
    print(f"\n{c}, all variants: most negative and most positive TFs")
    print(pd.concat([x.head(8), x.tail(8)])[["TF", "n", "r", "z", "ChIP share HepG2"]].to_string(index=False, float_format=fmt))
PT.to_csv(f"{OUT}/malinois_per_tf.csv", index=False); T5.to_csv(f"{OUT}/malinois_per_tf_summary.csv")

## 1.7 Reason 6: the pick among several motifs

In [ ]:
rows = []
for c in CELLS3:
    for s in ("all", "active"):
        g = V[sub(c, s) & V["has_call"]]
        one = g[g["n_calls"] == 1]
        y, cl = g[f"d_{c}"], g[f"clear_{c}"]
        rows.append({"cell": c, "variants": s, "called": len(g), "share with one call": len(one) / len(g),
                     "Pearson largest |alleleDiff|": pear(g["ad"], y), "Pearson summed alleleDiff": pear(g["sum_ad"], y),
                     "Pearson, one-call variants": pear(one["ad"], one[f"d_{c}"]),
                     "AUROC largest": auroc(cl, g["ad"].abs()), "AUROC summed": auroc(cl, g["sum_ad"].abs()),
                     "AUROC, one-call variants": auroc(one[f"clear_{c}"], one["ad"].abs())})
T6 = pd.DataFrame(rows).set_index(["cell", "variants"])
print(T6.to_string(float_format=fmt))
T6.to_csv(f"{OUT}/malinois_pick.csv")

---
# Section 2 — Kircher regions

## 2.1 Load: the same motifbreakR as `Pooled Regions 2` (its region 0 / 8 / 20 Pearson 0.177 / 0.279 / 0.464 must come back)

In [ ]:
ACCESSORY = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/accessary_files"
MPRA_PATH = f"{ACCESSORY}/mpra_sequences_with_embeddings_nt.pkl"
HOME = {"HepG2", "K562"}
EFFECT_TOP, P_CLEAR = 0.15, 1e-5
MBR_THRESH, MBR_NEUT = 0.85, 0.4
C = json.load(open(f"{WORK_DIR}/config.json"))
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
INFO = {int(k): v for k, v in MWJ["regions"].items()}


def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            mm = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(mm.group(1)) if mm else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = {"A": 0, "C": 1, "G": 2, "T": 3}
WIN = {}
for h in MWJ["hits"]:
    WIN.setdefault(h["region"], []).append(h)


def build(r, HS):
    # verbatim from Pooled Regions 2: SNV x hit rows and motifbreakR calls on the hits HS of region r
    S_ = f"{POOL}/regions/r{r}"
    hits = pd.read_csv(f"{S_}/hits.csv").set_index("hit_id")
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    q = q[q["hit_id"].isin(HS)].sort_values(["snv", "hit_id"]).reset_index(drop=True)
    sn = q["snv"].values
    st = np.r_[0, np.flatnonzero(sn[1:] != sn[:-1]) + 1]
    hi = np.searchsorted(HS, q["hit_id"].values)
    n_snv = len(st)
    MOT = hits.loc[HS, "motif"].values
    base = {}
    for h in WIN[r]:
        for i, b in enumerate(h["window"]):
            base.setdefault(h["start"] - C["motif_offset"] + i, b)
    lo = min(base)
    seq = np.array([ENC.get(base.get(x, "N"), 4) for x in range(lo, max(base) + 1)])
    s_ = snv.loc[sn[st]]
    pos, ref, alt = s_["local"].values - lo, s_["ref"].map(ENC).values, s_["alt"].map(ENC).values
    assert (seq[pos] == ref).all(), "rebuilt sequence disagrees with the SNV reference bases"
    out = []
    for m in sorted(set(MOT)):
        if m not in MBR_MATS:
            continue
        S, Sr, smin, smax = MBR_MATS[m]
        w = len(S)
        n = len(seq) - w + 1
        idx = np.arange(n)[:, None] + np.arange(w)[None, :]
        f_fwd, f_rev = S[np.arange(w), seq[idx]].sum(1), Sr[np.arange(w), seq[idx]].sum(1)
        stw = pos[:, None] - np.arange(w)[None, :]
        k = np.broadcast_to(np.arange(w), stw.shape)
        ok = (stw >= 0) & (stw < n)
        stc = np.clip(stw, 0, n - 1)
        ref_f, ref_r = np.where(ok, f_fwd[stc], -np.inf), np.where(ok, f_rev[stc], -np.inf)
        alt_f = ref_f + np.where(ok, S[k, alt[:, None]] - S[k, ref[:, None]], 0)
        alt_r = ref_r + np.where(ok, Sr[k, alt[:, None]] - Sr[k, ref[:, None]], 0)
        sr, sa = np.maximum(ref_f, ref_r).max(1), np.maximum(alt_f, alt_r).max(1)
        pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
        diff = sa - sr
        keep = np.isfinite(sr) & ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
        out += [(i, m, diff[i]) for i in np.flatnonzero(keep)]
    t = pd.DataFrame(out, columns=["i", "motif", "alleleDiff"])
    t["j"] = np.arange(len(t))
    rows_ = pd.DataFrame({"i": np.repeat(np.arange(n_snv), np.diff(np.r_[st, len(sn)])), "motif": MOT[hi], "h": hi})
    link = t[["j", "i", "motif"]].merge(rows_, on=["i", "motif"])
    keep = np.sort(link["j"].unique())
    t = t.loc[keep].reset_index(drop=True)
    return dict(snv=snv, n_snv=n_snv, snv_id=sn[st], y=snv.loc[sn[st], "log2FC"].values,
                DIFF=t["alleleDiff"].values, TI=t["i"].values, MOTP=t["motif"].values)


# p-values, if the MPRA file has them
_m = pd.read_pickle(MPRA_PATH)
_m = _m.drop(columns=[c for c in ("wt_embedding", "mut_embedding", "sequence_mut") if c in _m.columns])
PCOL = next((c for c in _m.columns if re.fullmatch(r"(?i)p[-_. ]?val(ue)?|p", str(c))), None)
print("MPRA file columns:", list(_m.columns))
print(f"p-value column: {PCOL!r}" + ("" if PCOL else "  -> only the top-15% definition of a clear effect"))
PV = {}
if PCOL:
    _m["region_id"] = _m["region_id"].astype(int)
    for rid, g in _m.groupby("region_id"):
        v = g[(g["alt"] != "-") & g["log2FC"].notna()].copy()
        v["local"] = v["pos"].astype(int) - v["start"].astype(int)
        v["ref"], v["alt"] = v["ref"].astype(str).str.upper(), v["alt"].astype(str).str.upper()
        PV[rid] = v.groupby(["local", "ref", "alt"])[PCOL].median()
del _m

KR = {}
for r in sorted(INFO):
    HS = np.array(sorted(h["hit_id"] for h in WIN.get(r, [])))
    if not len(HS):
        continue
    R = build(r, HS)
    snv = R["snv"].reset_index(drop=True)
    o = np.lexsort((-np.abs(R["DIFF"]), R["TI"]))
    fb = o[np.r_[True, R["TI"][o][1:] != R["TI"][o][:-1]]] if len(o) else o
    pos_of = pd.Series(np.arange(len(snv)), index=R["snv"].index)
    ad = np.zeros(len(snv)); called = np.zeros(len(snv), bool); inhit = np.zeros(len(snv), bool)
    inhit[pos_of[R["snv_id"]].values] = True
    ip = pos_of[R["snv_id"][R["TI"][fb]]].values
    ad[ip], called[ip] = R["DIFF"][fb], True
    y = snv["log2FC"].values
    p = (pd.MultiIndex.from_frame(snv[["local", "ref", "alt"]]).map(PV[r].to_dict().get) if r in PV else None)
    p = np.array([np.nan if v is None else v for v in p], float) if p is not None else np.full(len(snv), np.nan)
    KR[r] = dict(cell=INFO[r]["cell"], home=set(INFO[r]["cell"].split(",")) <= HOME, y=y, ad=ad, called=called,
                 inhit=inhit, p=p, clear15=np.abs(y) >= np.quantile(np.abs(y), 1 - EFFECT_TOP), clearp=p < P_CLEAR,
                 pear_hits=pear(ad[inhit], y[inhit]))
print("\ncheck against Pooled Regions 2 (motifbreakR SNV Pearson over SNVs in hits): "
      + ", ".join(f"region {r} {KR[r]['pear_hits']:.3f} (was {w})" for r, w in ((0, 0.177), (8, 0.279), (20, 0.464)) if r in KR))

## 2.2 Every region: coverage, effect rate, and motifbreakR's scores

In [ ]:
def region_row(r, K, which):
    cl = K[which]
    y, ad, called = K["y"], K["ad"], K["called"]
    clc = cl & called
    return {"region": r, "cell": K["cell"], "home": K["home"], "SNVs": len(y), "share called": called.mean(),
            "share clear": cl.mean(), "share of clear effects with a call": clc.sum() / max(cl.sum(), 1),
            "clear rate, called": cl[called].mean(), "clear rate, not called": cl[~called].mean(),
            "Pearson (SNVs in hits)": K["pear_hits"], "AUROC |alleleDiff| (all SNVs)": auroc(cl, np.abs(ad)),
            "AUROC among called": auroc(cl[called], np.abs(ad[called])),
            "sign right among clear called": (np.sign(ad[clc]) == np.sign(y[clc])).mean() if clc.any() else np.nan}


DEFS = [("top 15% |log2FC|", "clear15")] + ([("p < 1e-5", "clearp")] if PCOL else [])
KT = {}
for label, which in DEFS:
    KT[label] = pd.DataFrame([region_row(r, K, which) for r, K in KR.items()]).set_index("region")
    print(f"=== clear effect = {label}\n")
    print(KT[label].to_string(float_format=lambda x: f"{x:.3f}"))
    g = KT[label].drop(columns=["cell"]).groupby("home").mean(numeric_only=True)
    g.index = ["other cells" if not h else "HepG2/K562" for h in g.index]
    print("\nmean over regions\n" + g.drop(columns=["home", "SNVs"], errors="ignore").to_string(float_format=lambda x: f"{x:.3f}") + "\n")
    KT[label].to_csv(f"{OUT}/kircher_regions_{which}.csv")

## 2.3 Kircher: does a bigger motif change give a bigger effect?

Called SNVs pooled over regions, fifths of |alleleDiff|: |log2FC| percentile within its region, clear-effect rate
(top 15%), and sign agreement among clear effects. Split by HepG2/K562 regions vs the others.

In [ ]:
rows = []
for home in (True, False):
    ad = np.concatenate([K["ad"][K["called"]] for K in KR.values() if K["home"] == home])
    yp = np.concatenate([stats.rankdata(np.abs(K["y"]))[K["called"]] / len(K["y"]) * 100 for K in KR.values() if K["home"] == home])
    y = np.concatenate([K["y"][K["called"]] for K in KR.values() if K["home"] == home])
    cl = np.concatenate([K["clear15"][K["called"]] for K in KR.values() if K["home"] == home])
    q = pd.qcut(pd.Series(np.abs(ad)).rank(method="first"), 5, labels=False).values
    for b in range(5):
        x = q == b
        rows.append({"regions": "HepG2/K562" if home else "other cells", "|alleleDiff| fifth": b + 1, "n": int(x.sum()),
                     "mean |alleleDiff|": np.abs(ad[x]).mean(), "mean |log2FC| pct": yp[x].mean(),
                     "clear-effect rate": cl[x].mean(), "sign right among clear": (np.sign(ad[x & cl]) == np.sign(y[x & cl])).mean()})
T7 = pd.DataFrame(rows).set_index(["regions", "|alleleDiff| fifth"])
print(T7.to_string(float_format=lambda x: f"{x:.3f}"))
T7.to_csv(f"{OUT}/kircher_by_alleleDiff.csv")

---
# Section 3 — the two datasets side by side

Malinois per cell (all variants), Kircher pooled over HepG2/K562 regions and over the other regions (clear = top 15%).
"Clear rate ratio" = clear-effect rate of called / not-called variants: how much better than chance motifbreakR's
call is at finding an effect at all.

In [ ]:
rows = []
for c in CELLS3:
    m = sub(c, "all")
    g = V[m]
    cl, called, ad, y = g[f"clear_{c}"].values, g["has_call"].values, g["ad"].fillna(0).values, g[f"d_{c}"].values
    clc = cl & called
    rows.append({"dataset": f"Malinois {c}", "variants": len(g), "share called": called.mean(), "share clear": cl.mean(),
                 "share of clear with a call": clc.sum() / cl.sum(), "clear rate ratio": cl[called].mean() / cl[~called].mean(),
                 "AUROC among called": auroc(cl[called], np.abs(ad[called])), "Pearson among called": pear(ad[called], y[called]),
                 "sign right among clear called": (np.sign(ad[clc]) == np.sign(y[clc])).mean()})
for home in (True, False):
    Ks = [K for K in KR.values() if K["home"] == home]
    cl = np.concatenate([K["clear15"] for K in Ks]); called = np.concatenate([K["called"] for K in Ks])
    ad = np.concatenate([K["ad"] for K in Ks]); y = np.concatenate([(K["y"] - K["y"].mean()) / K["y"].std() for K in Ks])
    clc = cl & called
    rows.append({"dataset": f"Kircher {'HepG2/K562' if home else 'other cells'} ({len(Ks)} regions)", "variants": len(cl),
                 "share called": called.mean(), "share clear": cl.mean(), "share of clear with a call": clc.sum() / cl.sum(),
                 "clear rate ratio": cl[called].mean() / cl[~called].mean(),
                 "AUROC among called": auroc(cl[called], np.abs(ad[called])), "Pearson among called": pear(ad[called], y[called]),
                 "sign right among clear called": (np.sign(ad[clc]) == np.sign(y[clc])).mean()})
T8 = pd.DataFrame(rows).set_index("dataset")
print(T8.to_string(float_format=lambda x: f"{x:.3f}"))
T8.to_csv(f"{OUT}/side_by_side.csv")

## End

Outputs in `mbr_diagnosis/`: `malinois_ceiling.csv`, `malinois_coverage.csv`, `malinois_by_activity.csv`,
`malinois_by_alleleDiff.csv`, `malinois_per_tf.csv`, `malinois_per_tf_summary.csv`, `malinois_pick.csv`,
`kircher_regions_*.csv`, `kircher_by_alleleDiff.csv`, `side_by_side.csv`.